In [2]:
import requests
import pandas as pd
import os
import time

# Create folders if they don't exist yet
os.makedirs("../data/raw",   exist_ok=True)
os.makedirs("../data/clean", exist_ok=True)

print("✅ Folders ready:")
print("  ", os.path.abspath("../data/raw"))
print("  ", os.path.abspath("../data/clean"))
print()

# ── Helper function ──────────────────────────────────────────
BASE_URL = "https://api.data.gov.my/opendosm"

def fetch_dosm(dataset_id, limit=1000, extra_params=None):
    """
    Fetch one dataset from OpenDOSM API.
    Saves CSV to ../data/raw/  and  returns a DataFrame.
    """
    params = {"id": dataset_id, "limit": limit}
    if extra_params:
        params.update(extra_params)

    try:
        response = requests.get(BASE_URL, params=params, timeout=30)

        if response.status_code == 200:
            data = response.json()

            # Handle both {"data": [...]}  and  direct list responses
            if isinstance(data, dict) and "data" in data:
                df = pd.DataFrame(data["data"])
            elif isinstance(data, list):
                df = pd.DataFrame(data)
            else:
                print(f"⚠️  {dataset_id}: Unexpected response format")
                return None

            # Save to CSV
            filepath = f"../data/raw/{dataset_id}.csv"
            df.to_csv(filepath, index=False)
            print(f"  ✅  {dataset_id:<40} {len(df):>5} rows  →  saved")
            return df

        else:
            print(f"  ❌  {dataset_id:<40} HTTP {response.status_code}")
            return None

    except requests.exceptions.ConnectionError:
        print(f"  ❌  {dataset_id:<40} No internet connection")
        return None
    except requests.exceptions.Timeout:
        print(f"  ❌  {dataset_id:<40} Request timed out — try again")
        return None
    except Exception as e:
        print(f"  ❌  {dataset_id:<40} Error: {e}")
        return None

print("✅  fetch_dosm() function ready!")

# ── fetch_all_pages: for large datasets (population, cpi) ────
def fetch_all_pages(dataset_id):
    """
    Paginated fetch — use this for large datasets that exceed
    1000 rows (population_malaysia, population_state, etc.)
    """
    all_rows = []
    offset   = 0
    limit    = 1000
    page     = 1
    print(f"Downloading {dataset_id} (paginated)...")
    while True:
        params   = {"id": dataset_id, "limit": limit, "offset": offset}
        response = requests.get(BASE_URL, params=params, timeout=30)
        if response.status_code != 200:
            print(f"  ❌ HTTP {response.status_code}")
            break
        data = response.json()
        if isinstance(data, dict) and "data" in data:
            rows = data["data"]
        elif isinstance(data, list):
            rows = data
        else:
            break
        if len(rows) == 0:
            break
        all_rows.extend(rows)
        print(f"  Page {page} — {len(all_rows):,} rows so far")
        if len(rows) < limit:
            break
        offset += limit
        page   += 1
    df = pd.DataFrame(all_rows)
    df.to_csv(f"../data/raw/{dataset_id}.csv", index=False)
    print(f"  ✅ {dataset_id} — {len(df):,} rows saved!\n")
    return df

print("✅  fetch_all_pages() ready!")

✅ Folders ready:
   C:\Users\nizza\Documents\my-projects\malaysia-socioeconomic-portfolio\data\raw
   C:\Users\nizza\Documents\my-projects\malaysia-socioeconomic-portfolio\data\clean

✅  fetch_dosm() function ready!
✅  fetch_all_pages() ready!


In [ ]:
print("=" * 55)
print("TEMA 1 — CORE DATASETS (The Ageing Nation Story)")
print("=" * 55)

pop_malaysia    = fetch_all_pages("population_malaysia")
pop_state       = fetch_all_pages("population_state")
pop_district    = fetch_all_pages("population_district")
fertility_state = fetch_all_pages("fertility_state")

fertility  = fetch_dosm("fertility")
births     = fetch_dosm("births_annual")
deaths     = fetch_dosm("deaths")
marriages  = fetch_dosm("marriages")
hh_profile = fetch_dosm("hh_profile")

print()
print("✅  Tema 1 CORE done! Run Cell 3 next.")


TEMA 1 — CORE DATASETS (The Ageing Nation Story)
  Page 1 — 1,000 rows so far
  Page 2 — 2,000 rows so far
  Page 3 — 3,000 rows so far
  Page 4 — 4,000 rows so far
  Page 5 — 5,000 rows so far
  Page 6 — 6,000 rows so far
  Page 7 — 7,000 rows so far
  Page 8 — 8,000 rows so far
  Page 9 — 9,000 rows so far
  Page 10 — 10,000 rows so far
  Page 11 — 11,000 rows so far
  Page 12 — 12,000 rows so far
  Page 13 — 13,000 rows so far
  Page 14 — 14,000 rows so far
  Page 15 — 15,000 rows so far
  Page 16 — 16,000 rows so far
  Page 17 — 17,000 rows so far
  Page 18 — 18,000 rows so far
  Page 19 — 19,000 rows so far
  Page 20 — 20,000 rows so far
  Page 21 — 21,000 rows so far
  Page 22 — 22,000 rows so far
  Page 23 — 23,000 rows so far
  Page 24 — 24,000 rows so far
  Page 25 — 25,000 rows so far
  Page 26 — 26,000 rows so far
  Page 27 — 27,000 rows so far
  Page 28 — 28,000 rows so far
  Page 29 — 29,000 rows so far
  Page 30 — 30,000 rows so far
  Page 31 — 31,000 rows so far
  Page 3

In [ ]:
print("=" * 55)
print("TEMA 2 — CORE DATASETS (Income vs Inflation Story)")
print("=" * 55)

hh_income           = fetch_dosm("hh_income")
hh_income_state     = fetch_dosm("hh_income_state")
hh_inequality       = fetch_dosm("hh_inequality")
hh_inequality_state = fetch_dosm("hh_inequality_state")
hh_poverty          = fetch_dosm("hh_poverty")
hh_poverty_state    = fetch_dosm("hh_poverty_state")
hies_state          = fetch_dosm("hies_state")
cpi_annual          = fetch_dosm("cpi_annual")

# ← FIXED: large CPI datasets need pagination too
cpi_headline  = fetch_all_pages("cpi_headline")
cpi_inflation = fetch_all_pages("cpi_headline_inflation")
cpi_state     = fetch_all_pages("cpi_state")
cpi_lowincome = fetch_all_pages("cpi_lowincome")

print()
print("✅  Tema 2 CORE done! Run Cell 5 next.")


In [ ]:
print("=" * 65)
print("  DOWNLOAD SUMMARY — ALL 21 DATASETS")
print("=" * 65)

all_datasets = {
    # ── Tema 1 Core ──────────────────────────────────────────
    "population_malaysia"     : pop_malaysia,
    "population_state"        : pop_state,
    "population_district"     : pop_district,
    "fertility"               : fertility,
    "fertility_state"         : fertility_state,
    "births_annual"           : births,
    "deaths"                  : deaths,
    "marriages"               : marriages,
    "hh_profile"              : hh_profile,
    # ── Tema 2 Core ──────────────────────────────────────────
    "hh_income"               : hh_income,
    "hh_income_state"         : hh_income_state,
    "cpi_headline"            : cpi_headline,
    "cpi_headline_inflation"  : cpi_inflation,
    "hh_inequality"           : hh_inequality,
    "hh_inequality_state"     : hh_inequality_state,
    "hh_poverty"              : hh_poverty,
    "hh_poverty_state"        : hh_poverty_state,
    "hies_state"              : hies_state,
    "cpi_state"               : cpi_state,
    "cpi_lowincome"           : cpi_lowincome,
    "cpi_annual"              : cpi_annual,
}

success = 0
failed  = []

print(f"  {'DATASET':<40} {'ROWS':>6}  {'COLUMNS'}")
print("  " + "-" * 60)

for name, df in all_datasets.items():
    if df is not None and len(df) > 0:
        cols = list(df.columns)
        print(f"  ✅  {name:<38} {len(df):>5} rows  |  {cols}")
        success += 1
    else:
        print(f"  ❌  {name:<38} FAILED or EMPTY")
        failed.append(name)

print()

print("── population_malaysia (first 3 rows) ──")
display(pop_malaysia.head(3))

print("── hh_income (first 3 rows) ──")
display(hh_income.head(3))

print("── cpi_headline_inflation (first 3 rows) ──")
display(cpi_inflation.head(3))

print("── hh_inequality_state (first 3 rows) ──")
display(hh_inequality_state.head(3))
print("=" * 65)
print(f"  Downloaded: {success} / {len(all_datasets)} datasets")
if failed:
    print(f"  ⚠️  Failed:   {failed}")
    print(f"  → Download these manually from open.dosm.gov.my")
    print(f"    and save the CSV into:  ../data/raw/")
else:
    print(f"  🎉  ALL 21 datasets downloaded successfully!")
print("=" * 65)

In [ ]:
print("── population_malaysia (first 3 rows) ──")
display(pop_malaysia.head(3))

print("── hh_income (first 3 rows) ──")
display(hh_income.head(3))

print("── cpi_headline_inflation (first 3 rows) ──")
display(cpi_inflation.head(3))

print("── hh_inequality_state (first 3 rows) ──")
display(hh_inequality_state.head(3))